# E2.3 · Voluntary frameworks as your spine

**Function E — AI for GRC → The Regulatory & Compliance Lead**  ·  *Security of AI*

Builds on **[E2.2 · Horizontal AI regulation](https://spbreed.github.io/cyber-commons/lessons/E2.2.html)**.

| | |
|---|---|
| Open-source tooling | NIST AI RMF, OSCAL |
| Open-weight models | — |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The hook

Voluntary frameworks are the cheapest structural decision available: build one control set against a recognised spine, then map it outward to every regime that asks. The alternative is a control set per regulator.

## 2 · The framework

```
   one control set, mapped outward

                 +--------------------+
                 |  your control set  |
                 +---------+----------+
                           |
        +------------+-----+------+------------+
        v            v            v            v
     NIST AI RMF  ISO 42001   sector rule   customer DDQ

   the alternative is a control set per regulator, forever
```

Voluntary frameworks make a better spine than regulation, for two reasons that
have nothing to do with enthusiasm for standards.

**They are written as controls.** NIST AI RMF and ISO 42001 describe things you
*do*. Regulation describes outcomes you must achieve, which is harder to
operationalise and easier to satisfy on paper.

**They change more slowly than the law.** Building against a framework and
mapping outward to regulation means new regulation is a mapping exercise rather
than a programme.

The method: pick one spine with the best coverage of the controls you actually
operate, map outward, and be explicit about what the spine does **not** reach —
because every spine has gaps, and the gaps are where the sector overlay lives.

## 3 · Demo — which spine covers your control set best

In [ ]:
from collections import defaultdict

CONTROLS = {
 "AC-1": ("NIST AI RMF: GOVERN-1.2", "ISO 42001: 6.1", "EU AI Act: Art.14"),
 "AC-2": ("NIST AI RMF: MANAGE-2.2", "ISO 42001: 8.1"),
 "SB-1": ("NIST AI RMF: MANAGE-2.1", "ISO 27001: A.8.20"),
 "SB-2": ("EU AI Act: Art.14",),
 "EV-1": ("ISO 42001: 9.1", "EU AI Act: Art.12"),
 "EV-2": ("NIST AI RMF: MEASURE-2.3",),
 "DR-1": ("NIST AI RMF: MEASURE-2.4", "ISO 42001: 9.1"),
 "ST-1": ("EU AI Act: Art.14", "DORA: Art.11"),
}
by_fw = defaultdict(set)
for cid, fws in CONTROLS.items():
    for f in fws:
        by_fw[f.split(":")[0]].add(cid)

print(f"{'framework':18s}{'covers':>8}  controls")
print("-" * 62)
for fw, cids in sorted(by_fw.items(), key=lambda kv: -len(kv[1])):
    print(f"{fw:18s}{len(cids):>8}  {sorted(cids)}")

spine = max(by_fw, key=lambda f: len(by_fw[f]))
gaps = sorted(set(CONTROLS) - by_fw[spine])
print(f"\nbest spine: {spine} covering {len(by_fw[spine])}/{len(CONTROLS)}")
print(f"not reached by the spine: {gaps}")

## 4 · Where it breaks — one framework per regulation

In [ ]:
def per_regulation(controls):
    """Build a separate control set for each instrument. The usual approach."""
    sets = defaultdict(set)
    for cid, fws in controls.items():
        for f in fws:
            sets[f.split(":")[0]].add(cid)
    return sets

sets = per_regulation(CONTROLS)
total_implementations = sum(len(v) for v in sets.values())
distinct_controls = len(CONTROLS)
print(f"distinct controls actually needed : {distinct_controls}")
print(f"control implementations if built per-framework : {total_implementations}")
print(f"duplication factor : {total_implementations/distinct_controls:.1f}×")

print("\ncontrols claimed by more than one framework:")
for cid, fws in CONTROLS.items():
    if len(fws) > 1:
        print(f"   {cid}  {len(fws)} frameworks: {[f.split(':')[0] for f in fws]}")
print("\nBuilt separately, these drift: the ISO version of AC-1 and the AI Act")
print("version diverge, evidence is produced twice, and neither is trusted.")
assert total_implementations > distinct_controls

## 5 · The control — one spine, mapped outward, gaps named

In [ ]:
def spine_plan(controls, spine):
    covered = {c for c, fws in controls.items() if any(f.startswith(spine) for f in fws)}
    gaps = sorted(set(controls) - covered)
    secondary = defaultdict(list)
    for g in gaps:
        for f in controls[g]:
            secondary[f.split(":")[0]].append(g)
    return {"spine": spine, "covered": sorted(covered), "gaps": gaps,
            "secondary_sources": {k: v for k, v in secondary.items()
                                  if not k.startswith(spine)}}

plan = spine_plan(CONTROLS, "NIST AI RMF")
print(f"spine              {plan['spine']}")
print(f"covered by spine   {len(plan['covered'])}/{len(CONTROLS)}  {plan['covered']}")
print(f"gaps               {plan['gaps']}")
print("secondary sources needed for the gaps:")
for fw, cids in plan["secondary_sources"].items():
    print(f"   {fw:20s} supplies {cids}")

print("\nstatement for the assessor:")
print(f"   'We operate {len(CONTROLS)} AI controls, built against {plan['spine']}.")
print(f"    {len(plan['covered'])} map directly to it; {len(plan['gaps'])} come from")
print(f"    {list(plan['secondary_sources'])}. Each control produces one artefact,")
print("    which satisfies every clause it maps to.'")
assert plan["gaps"]

## What you just proved

NIST AI RMF covers the most controls (4 of 8) and is selected as the spine, leaving SB-2, EV-1 and ST-1 as gaps supplied by ISO 42001, the EU AI Act and DORA. Building per-framework would produce 14 control implementations for 8 distinct controls — a 1.8× duplication factor with controls claimed by several frameworks drifting apart.

## Your turn

Pick your spine and justify it in one sentence to an assessor. "It has the best coverage of the controls we actually operate" is far stronger than "it is the one our regulator mentioned".

---

**Next → [E2.4 · Sector overlays](https://spbreed.github.io/cyber-commons/lessons/E2.4.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/E2.3.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/E2.3.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*